# Podstawy uczenia maszynowego i perceptron

Notebook laboratoryjny do modułu:

- uczenie nadzorowane
- problem klasyfikacji
- liniowa separowalność
- perceptron Rosenblatta
- porównanie z regresją logistyczną

**Cele laboratorium**
1. Zrozumieć, jak działa perceptron.
2. Zaimplementować perceptron od zera.
3. Zwizualizować granicę decyzyjną.
4. Porównać perceptron z regresją logistyczną.
5. Zobaczyć ograniczenia modeli liniowych na przykładzie XOR.

## 1. Import bibliotek

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 2. Dane liniowo separowalne

Na początek generujemy prosty zbiór danych 2D z dwiema klasami.
Dzięki temu łatwo zobaczymy, jak model uczy się granicy decyzyjnej.

In [ ]:
X, y = make_blobs(
    n_samples=200,
    centers=2,
    n_features=2,
    cluster_std=1.2,
    random_state=42
)

# Perceptron klasycznie wygodnie zapisuje etykiety jako -1 i +1
y_perc = np.where(y == 0, -1, 1)

print("Kształt X:", X.shape)
print("Pierwsze 5 etykiet:", y_perc[:5])

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=y_perc, cmap="bwr", edgecolor="k", alpha=0.8)
plt.title("Dane liniowo separowalne")
plt.xlabel("Cecha 1")
plt.ylabel("Cecha 2")
plt.show()

## 3. Intuicja perceptronu

Perceptron wyznacza funkcję:

$$ \hat{y} = sign(w \cdot x + b) $$

gdzie:

- `x` to wektor cech,
- `w` to wagi modelu,
- `b` to bias,
- `sign(...)` zwraca klasę `-1` albo `+1`.

Uczenie odbywa się tylko wtedy, gdy model popełni błąd.
Aktualizacja wag ma postać:

```python
w = w + lr * y * x
b = b + lr * y
```

## 4. Implementacja perceptronu od zera

In [ ]:
class PerceptronScratch:
    def __init__(self, lr=0.1, epochs=20):
        self.lr = lr
        self.epochs = epochs
        self.w = None
        self.b = 0.0
        self.errors_per_epoch = []

    def fit(self, X, y):
        # TODO Implementacja algorytmu perceptronu
        return self

    def decision_function(self, X):
        return np.dot(X, self.w) + self.b

    def predict(self, X):
        scores = self.decision_function(X)
        return np.where(scores >= 0, 1, -1)

## 5. Uczenie modelu

In [ ]:
perceptron = PerceptronScratch(lr=0.1, epochs=25)
perceptron.fit(X, y_perc)

print("Wagi:", perceptron.w)
print("Bias:", perceptron.b)
print("Błędy w epokach:", perceptron.errors_per_epoch)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(perceptron.errors_per_epoch) + 1), perceptron.errors_per_epoch, marker="o")
plt.title("Liczba błędów w kolejnych epokach")
plt.xlabel("Epoka")
plt.ylabel("Liczba błędów")
plt.grid(True)
plt.show()

## 6. Wizualizacja granicy decyzyjnej

In [ ]:
def plot_decision_boundary(model, X, y, title="Granica decyzyjna"):
    x_min, x_max = X[:, 0].min() - 1.0, X[:, 0].max() + 1.0
    y_min, y_max = X[:, 1].min() - 1.0, X[:, 1].max() + 1.0

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, alpha=0.25, cmap="bwr")
    plt.contour(xx, yy, Z, levels=[0], colors="k", linewidths=2)
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", edgecolor="k", alpha=0.85)
    plt.title(title)
    plt.xlabel("Cecha 1")
    plt.ylabel("Cecha 2")
    plt.show()

In [ ]:
plot_decision_boundary(
    perceptron,
    X,
    y_perc,
    title="Perceptron - granica decyzyjna"
)

## 7. Ocena modelu

In [ ]:
y_pred_perc = perceptron.predict(X)
acc_perc = accuracy_score(y_perc, y_pred_perc)

print("Accuracy perceptronu:", round(acc_perc, 4))
print("\nMacierz pomyłek:")
print(confusion_matrix(y_perc, y_pred_perc))

## 8. Porównanie z regresją logistyczną

Regresja logistyczna to także model liniowy, ale zamiast twardej reguły decyzyjnej
zwraca prawdopodobieństwo przynależności do klasy.

In [ ]:
# Dla sklearn użyjemy etykiet 0/1
logreg = LogisticRegression()
logreg.fit(X, y)

y_pred_log = logreg.predict(X)
acc_log = accuracy_score(y, y_pred_log)

print("Accuracy regresji logistycznej:", round(acc_log, 4))

In [ ]:
class LogisticWrapper:
    def __init__(self, model):
        self.model = model

    def predict(self, X):
        pred = self.model.predict(X)
        return np.where(pred == 0, -1, 1)

plot_decision_boundary(
    LogisticWrapper(logreg),
    X,
    y_perc,
    title="Regresja logistyczna - granica decyzyjna"
)

## 9. Porównanie wyników

W wielu prostych problemach liniowo separowalnych oba modele będą działać dobrze.
Różnice pojawiają się w:
- interpretacji wyniku,
- stabilności uczenia,
- probabilistycznym charakterze regresji logistycznej.

In [ ]:
print("Perceptron - accuracy:", round(acc_perc, 4))
print("Logistic regression - accuracy:", round(acc_log, 4))

print("\nRaport klasyfikacji dla regresji logistycznej:")
print(classification_report(y, y_pred_log))

## 10. Dane nieliniowe – przykład XOR

To klasyczny przykład pokazujący ograniczenia perceptronu.
Jednej linii nie da się tu użyć do poprawnego rozdzielenia klas.

In [ ]:
X_xor = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

y_xor = np.array([-1, 1, 1, -1])

plt.figure(figsize=(6, 5))
plt.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap="bwr", edgecolor="k", s=120)
plt.title("Problem XOR")
plt.xlabel("Cecha 1")
plt.ylabel("Cecha 2")
plt.xlim(-0.2, 1.2)
plt.ylim(-0.2, 1.2)
plt.grid(True)
plt.show()

In [ ]:
perceptron_xor = PerceptronScratch(lr=0.1, epochs=20)
perceptron_xor.fit(X_xor, y_xor)

print("Błędy w epokach (XOR):", perceptron_xor.errors_per_epoch)
print("Predykcje perceptronu:", perceptron_xor.predict(X_xor))
print("Prawdziwe etykiety   :", y_xor)

In [ ]:
plot_decision_boundary(
    perceptron_xor,
    X_xor,
    y_xor,
    title="Perceptron na XOR - ograniczenie modelu liniowego"
)

## 11. Regresja logistyczna na XOR

Regresja logistyczna bez dodatkowych transformacji cech także jest modelem liniowym,
więc również nie rozwiązuje poprawnie problemu XOR.

In [ ]:
y_xor_01 = np.where(y_xor == -1, 0, 1)
logreg_xor = LogisticRegression()
logreg_xor.fit(X_xor, y_xor_01)

print("Predykcje regresji logistycznej:", logreg_xor.predict(X_xor))
print("Prawdziwe etykiety            :", y_xor_01)

In [ ]:
plot_decision_boundary(
    LogisticWrapper(logreg_xor),
    X_xor,
    y_xor,
    title="Regresja logistyczna na XOR"
)